# RetailNova · EDA sintético con narrativa tipo proyecto Telco

Este notebook reproduce la lógica de un proyecto de machine learning de churn tipo Telco, adaptado al caso RetailNova:
- limpieza de datos y validación de calidad,
- EDA descriptivo con foco en segmentos y riesgo,
- split train/test sin fuga,
- transformaciones por tipo de variable,
- comparación de modelos,
- interpretación del negocio y límites del análisis.

Objetivo: priorizar clientes con mayor riesgo de abandono para actuar antes de que el problema se materialice.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.show = lambda *args, **kwargs: None
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'generated').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data' / 'generated' / 'retailnova_case1_sintetico.csv'

print('Ruta del dataset:', DATA_PATH)
print('Existe:', DATA_PATH.exists())


In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('Duplicados:', df.duplicated().sum())
print('id_cliente único:', df['id_cliente'].is_unique)
print('Target present:', 'churn_6m' in df.columns)
print('\nTipos de columnas:')
print(df.dtypes.to_string())

df.head(3)


## 3. Carga del dataset y contrato de calidad

Se carga el CSV sintético y se revisan controles básicos de integridad: número de filas, columna objetivo, ausencia de duplicados y tipos de variables. El objetivo es comprobar que el dataset se comporte como una base de negocio real y no como una tabla aleatoria.


In [ ]:
# Limpieza y validación de calidad inicial
df = df.copy()
missing = df.isna().sum().sort_values(ascending=False)
print('Nulos por columna:')
print(missing[missing > 0].to_string())

assert df['churn_6m'].isin([0, 1]).all(), 'El target debe ser binario.'
assert df['id_cliente'].is_unique, 'Hay duplicados en id_cliente.'
print('Limpieza inicial completada.')


## 4. Exploración inicial y diagnóstico de relaciones

Esta fase mide la prevalencia del target, la composición por segmentos, la zona y la región, y el nivel de riesgo operativo. El objetivo es distinguir señales reales de ruido para no construir un modelo sobre una base con artefactos de negocio.


In [ ]:
# Visualización del target y desagregado por atributos clave
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

counts = df['churn_6m'].value_counts().sort_index()
axes[0].bar(['No churn', 'Churn'], counts.values, color=['#4CAF50', '#F44336'])
axes[0].set_title('Distribución del target')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center')

region_churn = df.groupby('region')['churn_6m'].mean().sort_values(ascending=False)
(region_churn * 100).plot(kind='bar', ax=axes[1], color='#5DA5DA')
axes[1].set_title('Churn por región (%)')
axes[1].set_ylabel('% churn')

segment_churn = df.groupby('segmento')['churn_6m'].mean().sort_values(ascending=False)
(segment_churn * 100).plot(kind='bar', ax=axes[2], color='#F39C12')
axes[2].set_title('Churn por segmento (%)')
axes[2].set_ylabel('% churn')

plt.tight_layout(); plt.show()

print('Prevalencia global de churn:')
print(f"{(df['churn_6m'].mean() * 100).round(2)}%")
print('\nChurn por región:')
print((region_churn * 100).round(2).to_string())
print('\nChurn por segmento:')
print((segment_churn * 100).round(2).to_string())


In [ ]:
# Análisis de variables numéricas y correlación
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_summary = df[num_cols].describe().T
print(num_summary[['count', 'mean', 'std', 'min', '50%', 'max']].round(2).to_string())

corr = df[num_cols].corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=False, mask=mask, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matriz de correlación de variables numéricas')
plt.tight_layout(); plt.show()

comparative = df.groupby('churn_6m')[['ingresos_mensuales', 'uso_datos_gb', 'minutos_mes', 'incidencias_tecnicas', 'latencia_ms', 'tasa_fallos', 'nps']].mean().T
print('Promedio por churn de variables clave:')
print(comparative.round(2).to_string())


## 5. Diagnóstico de calidad y servicios

Se revisan métricas operativas, la distribución del target y sus diferencias por región, segmento y plan. La intención es detectar si el problema de churn es estructural y repetible o si se debe principalmente a ruido.


In [ ]:
TARGET = 'churn_6m'
DROP_COLS = ['id_cliente', 'fecha_alta']
FEATURES = [c for c in df.columns if c not in DROP_COLS + [TARGET]]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print('Train rows:', len(X_train))
print('Test rows:', len(X_test))
print('Train churn rate:', round(y_train.mean(), 4))
print('Test churn rate:', round(y_test.mean(), 4))


In [ ]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print('Numéricas:', len(numeric_features))
print('Categóricas:', len(categorical_features))
print('Categóricas:', categorical_features)

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)
print('Shape transformada train:', X_train_prep.shape)
print('Shape transformada test:', X_test_prep.shape)


## 6. Transformaciones y modelado

En este punto ya se tiene la estructura de negocio entendida. La siguiente etapa normaliza la base para que los modelos comparables no se vean afectados por escalas distintas o por variables categóricas sin tratamiento.


In [ ]:
models = {
    'Baseline': DummyClassifier(strategy='most_frequent'),
    'Logistica': LogisticRegression(max_iter=800, class_weight='balanced', random_state=SEED),
    'RandomForest': RandomForestClassifier(
        n_estimators=80,
        max_depth=10,
        min_samples_leaf=25,
        class_weight='balanced_subsample',
        random_state=SEED,
        n_jobs=-1
    )
}

results = []
for name, model in models.items():
    pipe = Pipeline([
        ('preprocess', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        'modelo': name,
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds, zero_division=0),
        'f1': f1_score(y_test, preds, zero_division=0),
        'roc_auc': roc_auc_score(y_test, proba),
    })

res_df = pd.DataFrame(results).sort_values('f1', ascending=False).reset_index(drop=True)
print(res_df.to_string(index=False, formatters={
    'accuracy': lambda x: f'{x:.3f}',
    'precision': lambda x: f'{x:.3f}',
    'recall': lambda x: f'{x:.3f}',
    'f1': lambda x: f'{x:.3f}',
    'roc_auc': lambda x: f'{x:.3f}',
}))


## 7. Interpretación de negocio y cierre

La evidencia del dataset sintético sugiere que el riesgo no es aleatorio: hay disparidades por región, plan, calidad de experiencia y nivel de servicio. Esto habla de un problema operativo y de retención con relevancia comercial. El modelo debe usarse como sistema de priorización, no como sustituto del juicio del negocio.

Puntos clave para actuar:
- poner foco en clientes con mayor riesgo en regiones y planes más vulnerables;
- diseñar campañas de retención por segmento y nivel de servicio;
- usar la prioridad de atención, NPS y calidad de experiencia para guiar intervenciones tempranas;
- revisar la lógica de precios, SLA y soporte para reducir churn estructural.
